In [3]:
import pandas as pd
import numpy as np
import seaborn as sb
#ingest the raw customer data
df=pd.read_excel("Project_2_Neobank_Churn_Engagement_Data.xlsx")
print(df)
print(df.info())

      Customer_ID Signup_Date Snapshot_Month         Country  Age Income_Band  \
0     CUST-200000  2025-08-11     2026-05-01        Pakistan   70   Lower-Mid   
1     CUST-200001  2022-06-20     2026-05-01          Canada   41   Lower-Mid   
2     CUST-200002  2023-11-29     2026-05-01          Canada   51         Low   
3     CUST-200003  2022-10-15     2026-05-01  United Kingdom   45   Lower-Mid   
4     CUST-200004  2024-10-08     2026-05-01  United Kingdom   62         Low   
...           ...         ...            ...             ...  ...         ...   
2045  CUST-202045  2024-10-03     2026-05-01    Saudi Arabia   68      Middle   
2046  CUST-202046  2025-08-28     2026-05-01          Canada   54      Middle   
2047  CUST-202047  2022-08-07     2026-05-01  United Kingdom   71      Middle   
2048  CUST-202048  2024-12-28     2026-05-01             UAE   26         Low   
2049  CUST-202049  2024-01-18     2026-05-01          Canada   60   Lower-Mid   

     Occupation_Type Plan_T

In [4]:
df['Signup_Date'] = pd.to_datetime(df['Signup_Date'], errors='coerce')
df['Snapshot_Month'] = pd.to_datetime(df['Snapshot_Month'], errors='coerce')
# Clean data having special characters

balance_cols = ['Account_Balance', 'Monthly_Deposits']

for col in balance_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace('$', '', regex=False)
        .str.replace(',', '', regex=False)
        .str.strip()
    )

        
for col in balance_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2050 entries, 0 to 2049
Data columns (total 31 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   Customer_ID                 2050 non-null   object        
 1   Signup_Date                 2050 non-null   datetime64[ns]
 2   Snapshot_Month              2050 non-null   datetime64[ns]
 3   Country                     2050 non-null   object        
 4   Age                         2050 non-null   int64         
 5   Income_Band                 2050 non-null   object        
 6   Occupation_Type             2050 non-null   object        
 7   Plan_Type                   2050 non-null   object        
 8   KYC_Status                  2050 non-null   object        
 9   Tenure_Months               2050 non-null   int64         
 10  Account_Balance             2050 non-null   float64       
 11  Monthly_Deposits            2050 non-null   float64     

In [5]:
#Step2
#flag, where Monthly_Transactions > 0 but App_Sessions_30D == 0 (impossible account execution). 


impossible_account_execution=(df['Monthly_Transactions'] > 0) & (df['App_Sessions_30D'] ==0)

quarantined_df = df[impossible_account_execution].copy()

# 3. Keep only valid records in your main DataFrame
clean_df = df[~impossible_account_execution].copy()  # ~ means "NOT"

# 4. Export corrupted records to CSV for auditing
quarantined_df.to_csv('quarantined_churn_data.csv', index=False)

# Optional: Inspect results
print(f"Quarantined rows: {len(quarantined_df)}")
print(f"Clean rows: {len(clean_df)}")

Quarantined rows: 45
Clean rows: 2005


In [6]:
#	"Core_Feature_Score"<30→"High" 
#	"Core_Feature_Score"<70→"Medium" 
#	"Else"→"Low" 

#clean_df['Core_Feature_Score'] = df['Core_Feature_Score']

clean_df['Core_Feature_Score'] = (
    df['Core_Feature_Score']
    .astype(str)
    .str.replace(r'[^\d.-]', '', regex=True)
)

# Convert the column to numeric, as the column is string type
clean_df['Core_Feature_Score'] = pd.to_numeric(clean_df['Core_Feature_Score'], errors='coerce')

conditions=[
    clean_df['Core_Feature_Score'] <30,
    clean_df['Core_Feature_Score'] <70
]

choices=['High', 'Medium']

clean_df['AI_Risk_Band'] = np.select(conditions, choices, default='Low')

print(clean_df[['Core_Feature_Score', 'AI_Risk_Band']].head(20))

    Core_Feature_Score AI_Risk_Band
0                   96          Low
1                   39       Medium
2                   83          Low
3                   45       Medium
4                   70          Low
5                   64       Medium
6                  100          Low
7                   94          Low
8                   41       Medium
9                   38       Medium
10                  60       Medium
11                   4         High
13                  88          Low
14                  17         High
15                  61       Medium
16                 100          Low
17                  87          Low
19                  72          Low
20                  39       Medium
21                 100          Low


In [7]:
#1. Making sure numeric types for count columns to avoid errors

clean_df['Failed_Logins'] = pd.to_numeric(clean_df['Failed_Logins'], errors='coerce').fillna(0)
clean_df['Support_Tickets'] = pd.to_numeric(clean_df['Support_Tickets'], errors='coerce').fillna(0)

#2. Calculate raw score (start at 100)
clean_df['Customer_Friction_Score'] = np.maximum(0,100 
    + (clean_df['Plan_Type'] == 'Premium') * 10
    - (clean_df['KYC_Status'] == 'Pending') * 40
    - clean_df['Failed_Logins'] * 25
    - clean_df['Support_Tickets'] * 5
)

# 3. Check the result
print(clean_df[['Plan_Type', 'Failed_Logins', 'KYC_Status', 'Support_Tickets', 'Customer_Friction_Score']].head(10))


  Plan_Type  Failed_Logins KYC_Status  Support_Tickets  \
0  Business              1  Completed                0   
1      Free              3  Completed                0   
2   Premium              2  Completed                0   
3      Free              0    Pending                0   
4      Free              1    Pending                0   
5  Business              1   Rejected                0   
6      Plus              0    Pending                0   
7   Premium              1  Completed                1   
8      Free              0  Completed                0   
9      Free              0    Pending                0   

   Customer_Friction_Score  
0                       75  
1                       25  
2                       60  
3                       60  
4                       35  
5                       75  
6                       60  
7                       80  
8                      100  
9                       60  


In [8]:
#1 # 1. Clean column and ensure numeric type

clean_df['Monthly_Transactions']=pd.to_numeric(clean_df['Monthly_Transactions'],errors='coerce')

# 2. Group by Plan_Type and compute CV scaled by 100 directly into the new column
std_series = clean_df.groupby('Plan_Type')['Monthly_Transactions'].transform('std')
mean_series = clean_df.groupby('Plan_Type')['Monthly_Transactions'].transform('mean')

clean_df['Plan_Transaction_Volatility_Index'] = (std_series / mean_series) * 100

# Verify result
print(clean_df[['Plan_Type', 'Monthly_Transactions', 'Plan_Transaction_Volatility_Index']].head(10))

  Plan_Type  Monthly_Transactions  Plan_Transaction_Volatility_Index
0  Business                    28                          36.664941
1      Free                    14                          46.309014
2   Premium                    26                          28.176193
3      Free                    16                          46.309014
4      Free                    25                          46.309014
5  Business                    26                          36.664941
6      Plus                    37                          38.847763
7   Premium                    37                          28.176193
8      Free                    15                          46.309014
9      Free                    16                          46.309014


In [9]:
#Step 5: Database Loading

from sqlalchemy import create_engine

print("Rows to upload:", len(clean_df)) 

# 2. Connect to MySQL 

USER = "root"
PASSWORD = "1234"  
HOST = "localhost"
PORT = "3306"
DATABASE = "Customer_Churn_Prediction_db"              
TABLE_NAME = "neobank_customer_churn"

engine = create_engine(f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DATABASE}")

# 3. Push it directly to your MySQL Server
print("Uploading data to MySQL...")
df.to_sql(
    name=TABLE_NAME,
    con=engine, 
    index=False, 
    if_exists='replace',
    chunksize=1000,      
    method='multi'
)

print("Success! Check MySQL Workbench and refresh your tables schema.")

Rows to upload: 2005
Uploading data to MySQL...
Success! Check MySQL Workbench and refresh your tables schema.
